## Linear Probing — Layer-wise Signal Localization

**Purpose**
- Probe WHERE in the encoder the readmission signal emerges.
Train logistic regression at each transformer layer to measure linear separability.
- Compare JEPA (softmax-free) vs. supervised (softmax) probing curves. **If JEPA representations are equally separable without softmax, the latent space organizes for the task geometry naturally.**

**Tasks**:
1. Load JEPA checkpoint, extract layer representations
2. Train probes at each layer, plot probing curve
3. Softmax baseline comparison (if available)
4. F3x sub-block stratification
5. Save results

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.utils.io import EXPERIMENTS_DIR, PROCESSED_DIR, load_sequences_dict
from src.utils.seed import SEED, load_seed, set_global_seed, get_rng

%matplotlib inline

In [ ]:
MODEL_TAG        = "test_01"
CHECKPOINT_NAME  = "checkpoint_40.pt"
SAVE_FIGS        = False
SAVE_DATA        = True

model_dir = EXPERIMENTS_DIR / MODEL_TAG
set_global_seed(load_seed(model_dir))
rng = get_rng()

ckt_path = model_dir / "checkpoints" / CHECKPOINT_NAME
res_dir = model_dir / "probing"
res_dir.mkdir(parents=True, exist_ok=True)

fig_dir  = model_dir / "figures"
def _sp(name: str):
    return fig_dir / name if SAVE_FIGS else None

print(f"Model dir:   {model_dir}")

### Load JEPA

In [ ]:
import torch
from torch.utils.data import DataLoader

from src.training.checkpoint import load_model_notrain
from src.training.dataset import JEPADataset, collate_fn
from src.utils.io import load_sequences

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load model, frozen
model, ckpt = load_model_notrain(ckt_path, device, restore_rng=False)
model.eval()
model.requires_grad_(False)

n_layers = len(model.context_encoder.layers)
embed_dim = model.embed_dim
print(f"Model: {n_layers} layers, embed_dim={embed_dim}")

# Load data and build dataloader
vocab_path = model_dir / "vocab.json"
with open(vocab_path) as f:
    vocab = json.load(f)

patients = load_sequences(n=0)  # all patients
pad_idx = vocab["[PAD]"]

dataset = JEPADataset(patients, vocab, pad_idx)
loader = DataLoader(dataset, batch_size=64, shuffle=False,
                    collate_fn=collate_fn,
                    num_workers=0)

print(f"Dataset: {len(dataset)} samples")
print(f"Vocab:   {len(vocab)} tokens")

### Extract Layer Representations

In [ ]:
from src.analysis.probing import extract_layer_representations

print("Extracting layer representations...")
layer_reps = extract_layer_representations(model, loader, device)

print(f"\nExtracted {layer_reps['n_layers']} layers + final")
for key in [f"layer_{i}" for i in range(layer_reps['n_layers'])] + ["final"]:
    print(f"  {key:12s}  shape={layer_reps[key].shape}")

labels = layer_reps["labels"]
subject_ids = layer_reps["subject_ids"]
n_pos = int(labels.sum())
print(f"\nLabels: {len(labels)} samples, {n_pos} positive ({n_pos/len(labels):.1%})")

### Linear Probing Sweep

In [ ]:
from src.training.probe.train_probe import run_probing_sweep

print("Running probing sweep (5-fold stratified CV)...\n")
sweep = run_probing_sweep(layer_reps, cv=5, seed=SEED)

print(f"\nBest layer: {sweep['best_layer']} (AUC={sweep['best_auc']:.4f})")
print(f"\nInterpretation:\n  {sweep['interpretation']}")

In [ ]:
from src.analysis.plotting import plot_probing_curve
plot_probing_curve(sweep, save_path=_sp("probing_curve.png"))

In [ ]:
from src.analysis.plotting import plot_probing_metrics
plot_probing_metrics(sweep, save_path=_sp("probing_metrics.png"))

### EXTRA: Softmax Baseline Comparison

To complete this comparison, train a separate model with a softmax classification
head on the same architecture. Then extract layer representations and run the
same probing sweep.

**Tasks:**
1. Take the same TransformerEncoder architecture
2. Add `nn.Linear(embed_dim, 2)` classification head
3. Train end-to-end with cross-entropy loss on the readmission label
4. Extract layer representations using `extract_layer_representations`
5. Run `run_probing_sweep` on those representations
6. Compare using `compare_softmax_baseline` and `plot_probing_comparison`

In [ ]:
# Uncomment and fill in when softmax model is available:

# from src.analysis.probing import compare_softmax_baseline
# from src.analysis.plotting import plot_probing_comparison
#
# softmax_model, _ = load_model_from_checkpoint(softmax_ckpt_path, device)
# softmax_model.eval()
# softmax_model.requires_grad_(False)
#
# softmax_reps = extract_layer_representations(softmax_model, loader, device)
# softmax_sweep = run_probing_sweep(softmax_reps, cv=5, seed=SEED)
#
# comparison = compare_softmax_baseline(sweep, softmax_sweep)
# print(comparison["interpretation"])
#
# plot_probing_comparison(
#     sweep, softmax_sweep, show=True,
#     save_path=res_dir / "probing_comparison.png",
# )

print("Softmax baseline: not yet trained. See protocol above.")

## Section 4: F3x Sub-block Stratification

Does probe accuracy differ for F31 (bipolar) vs F32 (depressive) vs F33 (recurrent)?
This connects back to the Tier 2 metadata features.

In [ ]:
from src.training.probe.train_probe import train_linear_probe

sequences_path = PROCESSED_DIR / "sequences.jsonl"
patients_dict = load_sequences_dict(sequences_path)

# Build F3x sub-block labels for each sample
f3x_blocks = {
    "F31 (bipolar)": "F31",
    "F32 (depressive)": "F32",
    "F33 (recurrent)": "F33",
}


def has_f3x_code(patient_dict, prefix):
    """Check if any encounter has an ICD code starting with prefix."""
    for enc in patient_dict.get("encounters", []):
        for code in enc.get("icd_codes", []):
            if str(code).upper().startswith(prefix):
                return True
    return False


# Use final-layer representations for stratification
X_final = layer_reps["final"]

stratified_results = {}
for block_name, prefix in f3x_blocks.items():
    # Build binary label: 1 if patient has this F3x code, 0 otherwise
    block_labels = np.zeros(len(subject_ids), dtype=int)
    for i, sid in enumerate(subject_ids):
        p = patients_dict.get(str(sid))
        if p and has_f3x_code(p, prefix):
            block_labels[i] = 1

    n_pos = int(block_labels.sum())
    n_neg = len(block_labels) - n_pos
    print(f"\n{block_name}: {n_pos} positive, {n_neg} negative")

    if n_pos < 10:
        print(f"  Skipping — too few positive samples ({n_pos})")
        continue

    result = train_linear_probe(X_final, block_labels, cv=5, seed=SEED)
    stratified_results[block_name] = result
    print(f"  AUC={result['mean_auc']:.4f} +/- {result['std_auc']:.4f}  "
          f"F1={result['mean_f1']:.4f}  acc={result['mean_accuracy']:.4f}")

In [ ]:
if stratified_results:
    from src.analysis.plotting import plot_f3x_stratified_probing

    plot_f3x_stratified_probing(
        stratified_results, show=True,
        save_path=res_dir / "f3x_probing.png",
    )
else:
    print("No F3x sub-blocks had enough positive samples for probing.")

## Section 5: Save Results

In [ ]:
# Save probing sweep results
sweep_save = {
    "per_layer": sweep["per_layer"],
    "best_layer": sweep["best_layer"],
    "best_auc": sweep["best_auc"],
    "interpretation": sweep["interpretation"],
    "summary": [(k, auc, std) for k, auc, std in sweep["summary"]],
}

with open(res_dir / "probing_sweep.json", "w") as f:
    json.dump(sweep_save, f, indent=2)

# Save stratified results
if stratified_results:
    with open(res_dir / "f3x_stratified.json", "w") as f:
        json.dump(stratified_results, f, indent=2)

# Save layer representations for reuse
# np.savez_compressed(
#     res_dir / "layer_representations.npz",
#     **{k: v for k, v in layer_reps.items() if isinstance(v, np.ndarray)},
#     n_layers=np.array(layer_reps["n_layers"]),
# )

print(f"Results saved -> {res_dir}")
for p in sorted(res_dir.iterdir()):
    print(f"  {p.name}")

**Linear Probing Summary**

In [ ]:
print(f"\nModel: {MODEL_TAG} ({n_layers} layers, embed_dim={embed_dim})")
print(f"Checkpoint: {CHECKPOINT_NAME}")
print("=" * 60)
print(f"Samples: {len(labels)} ({n_pos} positive, {n_pos/len(labels):.1%})")
print(f"\nProbing curve (AUC by layer):")
for key, auc, std in sweep["summary"]:
    marker = " <-- best" if key == sweep["best_layer"] else ""
    print(f"  {key:12s}  {auc:.4f} +/- {std:.4f}{marker}")
print(f"\n{sweep['interpretation']}")

if stratified_results:
    print(f"\nF3x sub-block probing (final layer):")
    for name, res in stratified_results.items():
        print(f"  {name:25s}  AUC={res['mean_auc']:.4f}  (n_pos={res['n_positive']})")